In [1]:
import pandas as pd
import glob
import time
import re

In [57]:
caminho_da_pasta = '..\data\\raw\Dados_SINAN\\'

In [ ]:
colunas_para_usar = ['DT_NOTIFIC', 'SG_UF_NOT']

# Definição de chunck para processar os dados em parte
tamanho_do_chunk = 100000

# Lista para armazenar os resultados agregados de cada pedaço processado.
lista_de_resultados_agregados = []

In [ ]:
print("\n--- Iniciando o Processamento dos Arquivos ---")

# Encontra todos os arquivos com a extensão DENGRBR*.csv na pasta especificada
todos_arquivos = glob.glob(caminho_da_pasta + "DENGBR*.csv")

if not todos_arquivos:
    print(f"AVISO: Nenhum arquivo .csv encontrado na pasta: {caminho_da_pasta}")
    print("Por favor, verifique se o caminho está correto e se os arquivos estão lá.")
else:
    print(f"Encontrados {len(todos_arquivos)} arquivos para processar.")




--- Iniciando o Processamento dos Arquivos ---
Encontrados 12 arquivos para processar.


In [ ]:
# Processamento de todos os arquivos para agrupar em um arquivo único
for arquivo in todos_arquivos:
    print(f"\n--- Processando o arquivo: {arquivo} ---")
    start_time = time.time()

    chunk_iterator = pd.read_csv(
        arquivo,
        sep=',',
        encoding='utf-8',
        usecols=colunas_para_usar,
        chunksize=tamanho_do_chunk,
        low_memory=False
    )

    for i, chunk in enumerate(chunk_iterator):
        print(f"\n--- Processando o chunk {i + 1} ---")

        # Limpeza: remove linhas onde as colunas essenciais são nulas
        chunk.dropna(subset=colunas_para_usar, inplace=True)

        # Engenharia de Features: converte data e extrai Ano/Mês
        chunk[colunas_para_usar[0]] = pd.to_datetime(chunk[colunas_para_usar[0]], errors='coerce')

        chunk['Ano'] = chunk[colunas_para_usar[0]].dt.year
        chunk['Mês'] = chunk[colunas_para_usar[0]].dt.month

        # agrupa e conta os casos dentro do chunk
        resultado_chunk = chunk.groupby(['Ano', 'Mês', 'SG_UF_NOT']).size().reset_index(name='Quantidade de Casos')

        # Guarda o resultado agregado do chunk na lista principal
        lista_de_resultados_agregados.append(resultado_chunk)

    end_time = time.time()
    print(f"Arquivo finalizado em {end_time - start_time:.2f} segundos.")





--- Processando o arquivo: C:\Users\55219\Desktop\Lucas\TI\Arquivos\Projetos_aprendizado\TechChallenge_Fase3_Dengue\data\raw\Dados_SINAN\DENGBR14.csv ---

--- Processando o chunk 1 ---

--- Processando o chunk 2 ---

--- Processando o chunk 3 ---

--- Processando o chunk 4 ---

--- Processando o chunk 5 ---

--- Processando o chunk 6 ---

--- Processando o chunk 7 ---

--- Processando o chunk 8 ---

--- Processando o chunk 9 ---

--- Processando o chunk 10 ---
Arquivo finalizado em 5.87 segundos.

--- Processando o arquivo: C:\Users\55219\Desktop\Lucas\TI\Arquivos\Projetos_aprendizado\TechChallenge_Fase3_Dengue\data\raw\Dados_SINAN\DENGBR15.csv ---

--- Processando o chunk 1 ---

--- Processando o chunk 2 ---

--- Processando o chunk 3 ---

--- Processando o chunk 4 ---

--- Processando o chunk 5 ---

--- Processando o chunk 6 ---

--- Processando o chunk 7 ---

--- Processando o chunk 8 ---

--- Processando o chunk 9 ---

--- Processando o chunk 10 ---

--- Processando o chunk 11 ---

In [ ]:
if lista_de_resultados_agregados:
    print("\n--- Todos os arquivos foram processados. Consolidando o resultado final... ---")

    # Concatena todos os DataFrames de resultados agregados em um só
    df_agregado = pd.concat(lista_de_resultados_agregados)

    # Faz um último 'groupby' para somar os valores de diferentes chunks/arquivo que pertencem ao mesmo Ano, Mês e UF.
    df_resultado_final = df_agregado.groupby(['Ano', 'Mês', 'SG_UF_NOT'])['Quantidade de Casos'].sum().reset_index()

    # Limpeza final para garantir tipos de dados corretos
    df_resultado_final.dropna(subset=['Ano', 'Mês'], inplace=True)
    df_resultado_final['Ano'] = df_resultado_final['Ano'].astype(int)
    df_resultado_final['Mês'] = df_resultado_final['Mês'].astype(int)

    print("\n--- DataFrame Final Consolidado ---")
    print(df_resultado_final.sort_values(by=['Ano', 'Mês']))

    # Salvar o resultado final em um novo arquivo CSV no na mesma pasta
    caminho_salvar = '..\data\\raw\Dados_SINAN\\resultado_consolidado_dengue_2014_2025.csv'
    df_resultado_final.to_csv(caminho_salvar, index=False)
    print(f"\nResultado final salvo em: {caminho_salvar}")
else:
    print("\nProcessamento concluído, mas nenhum dado foi gerado. Verifique os arquivos de origem.")



--- Todos os arquivos foram processados. Consolidando o resultado final... ---

--- DataFrame Final Consolidado ---
       Ano  Mês  SG_UF_NOT  Quantidade de Casos
0     2013   12         11                   48
1     2013   12         12                   37
2     2013   12         13                   56
3     2013   12         14                   12
4     2013   12         15                   34
...    ...  ...        ...                  ...
3715  2025    7         43                  355
3716  2025    7         50                  199
3717  2025    7         51                  158
3718  2025    7         52                  546
3719  2025    7         53                  174

[3720 rows x 4 columns]

Resultado final salvo em: C:\Users\55219\Desktop\Lucas\TI\Arquivos\Projetos_aprendizado\TechChallenge_Fase3_Dengue\data\raw\Dados_SINAN\resultado_consolidado_dengue_2014_2025.csv


In [ ]:
df_resultado_final['Ano'].unique() # verifica os anos presentes no dataset final

array([2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023,
       2024, 2025])

In [ ]:
df_resultado_final.sort_values(by=['Ano', 'Mês']) # visualiza o dataset final ordenado por Ano e Mês

,Ano,Mês,SG_UF_NOT,Quantidade de Casos
0,2013,12,11,48
1,2013,12,12,37
2,2013,12,13,56
3,2013,12,14,12
4,2013,12,15,34
...,...,...,...,...
3715,2025,7,43,355
3716,2025,7,50,199
3717,2025,7,51,158
3718,2025,7,52,546


In [ ]:
df_resultado_final.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3720 entries, 0 to 3719
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   Ano                  3720 non-null   int64
 1   Mês                  3720 non-null   int64
 2   SG_UF_NOT            3720 non-null   int64
 3   Quantidade de Casos  3720 non-null   int64
dtypes: int64(4)
memory usage: 116.4 KB


In [ ]:
# Mapeamento de códigos IBGE para siglas de estados
mapa_codigo_para_uf = {
    12: 'AC', 27: 'AL', 16: 'AP', 13: 'AM', 29: 'BA', 23: 'CE',
    53: 'DF', 32: 'ES', 52: 'GO', 21: 'MA', 51: 'MT', 50: 'MS',
    31: 'MG', 15: 'PA', 25: 'PB', 41: 'PR', 26: 'PE', 22: 'PI',
    24: 'RN', 43: 'RS', 33: 'RJ', 11: 'RO', 14: 'RR', 42: 'SC',
    35: 'SP', 28: 'SE', 17: 'TO'
}

In [ ]:
# Adiciona a coluna COD_UF com base no mapeamento e exclui a coluna SG_UF_NOT
df_resultado_final['COD_UF'] = df_resultado_final['SG_UF_NOT'].map(mapa_codigo_para_uf)
df_resultado_final.drop(columns=['SG_UF_NOT'], inplace=True)

# Cria a coluna 'periodo' no formato period e remove as colunas 'Ano' e 'Mês'
df_resultado_final["periodo"] = pd.to_datetime(dict(year=df_resultado_final["Ano"], month=df_resultado_final["Mês"], day=1)).dt.to_period("M")
df_resultado_final.drop(columns=['Ano', 'Mês'], inplace=True)

df_resultado_final.head()

,Quantidade de Casos,COD_UF,periodo
0,48,RO,2013-12
1,37,AC,2013-12
2,56,AM,2013-12
3,12,RR,2013-12
4,34,PA,2013-12


In [27]:
df_resultado_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3720 entries, 0 to 3719
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype    
---  ------               --------------  -----    
 0   Quantidade de Casos  3720 non-null   int64    
 1   COD_UF               3720 non-null   object   
 2   periodo              3720 non-null   period[M]
dtypes: int64(1), object(1), period[M](1)
memory usage: 87.3+ KB


In [ ]:
# Cria uma tabela pivô para visualizar os dados com 'periodo' como índice e 'COD_UF' como colunas
df_pivot = df_resultado_final.pivot(index="periodo", columns="COD_UF", values="Quantidade de Casos")
df_pivot.info()

<class 'pandas.core.frame.DataFrame'>
PeriodIndex: 140 entries, 2013-12 to 2025-07
Freq: M
Data columns (total 27 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AC      140 non-null    float64
 1   AL      140 non-null    float64
 2   AM      140 non-null    float64
 3   AP      140 non-null    float64
 4   BA      140 non-null    float64
 5   CE      140 non-null    float64
 6   DF      140 non-null    float64
 7   ES      81 non-null     float64
 8   GO      140 non-null    float64
 9   MA      139 non-null    float64
 10  MG      140 non-null    float64
 11  MS      140 non-null    float64
 12  MT      140 non-null    float64
 13  PA      140 non-null    float64
 14  PB      140 non-null    float64
 15  PE      140 non-null    float64
 16  PI      140 non-null    float64
 17  PR      140 non-null    float64
 18  RJ      140 non-null    float64
 19  RN      140 non-null    float64
 20  RO      140 non-null    float64
 21  RR      140 non-null

In [ ]:
# Verifica os valores NaN na coluna 'ES' do DataFrame pivô e quais anos não estão presentes
df_pivot[df_pivot['ES'].isna()]['ES']

periodo
2020-05   NaN
2020-06   NaN
2020-07   NaN
2020-08   NaN
2020-09   NaN
2020-10   NaN
2020-11   NaN
2020-12   NaN
2021-01   NaN
2021-02   NaN
2021-03   NaN
2021-04   NaN
2021-05   NaN
2021-06   NaN
2021-07   NaN
2021-08   NaN
2021-09   NaN
2021-10   NaN
2021-11   NaN
2021-12   NaN
2022-01   NaN
2022-02   NaN
2022-03   NaN
2022-04   NaN
2022-05   NaN
2022-06   NaN
2022-07   NaN
2022-08   NaN
2022-09   NaN
2022-10   NaN
2022-11   NaN
2022-12   NaN
2023-01   NaN
2023-02   NaN
2023-03   NaN
2023-04   NaN
2023-05   NaN
2023-06   NaN
2023-07   NaN
2023-08   NaN
2023-09   NaN
2023-10   NaN
2023-11   NaN
2023-12   NaN
2024-01   NaN
2024-02   NaN
2024-03   NaN
2024-04   NaN
2024-05   NaN
2024-06   NaN
2024-07   NaN
2024-08   NaN
2024-09   NaN
2024-10   NaN
2024-11   NaN
2024-12   NaN
2025-04   NaN
2025-06   NaN
2025-07   NaN
Freq: M, Name: ES, dtype: float64

In [28]:
# Inclusão de dados do ES manualmente por meio de dicionário
# Foi utilizado o http://tabnet.datasus.gov.br/cgi/deftohtm.exe?sinannet/cnv/denguebes.def

# Verificar se ES já possui dados entre 2020-05 e 2024-12
periodo_alvo = pd.period_range("2020-05", "2024-12", freq="M")
es_faltando = ~df_resultado_final[(df_resultado_final["COD_UF"] == "ES")]["periodo"].isin(periodo_alvo)

if es_faltando.all():
    # Dados fornecidos manualmente com base em consulta 
    dados_es = {
        "2020": {"Maio": 4, "Julho": 1, "Agosto": 3, "Outubro": 1, "Novembro": 5, "Dezembro": 467},
        "2021": {"Janeiro": 4, "Marco": 4, "Abril": 15, "Maio": 1, "Junho": 1, "Julho": 4, "Dezembro": 3},
        "2022": {"Janeiro": 3, "Fevereiro": 3, "Marco": 6, "Abril": 14, "Maio": 14, "Junho": 3, "Julho": 5,
                 "Agosto": 2, "Setembro": 2, "Outubro": 5, "Novembro": 2, "Dezembro": 7},
        "2023": {"Janeiro": 29, "Fevereiro": 26, "Marco": 58, "Abril": 35, "Maio": 40, "Junho": 23, "Julho": 9,
                 "Agosto": 4, "Setembro": 6, "Outubro": 6, "Novembro": 7, "Dezembro": 9},
        "2024": {"Janeiro": 62, "Fevereiro": 133, "Marco": 142, "Abril": 105, "Maio": 58, "Junho": 21, "Julho": 11,
                 "Agosto": 9, "Setembro": 7, "Outubro": 6, "Novembro": 7, "Dezembro": 14}
    }

    # Mapeamento de meses
    meses_map = {
        "Janeiro": 1, "Fevereiro": 2, "Marco": 3, "Abril": 4, "Maio": 5, "Junho": 6,
        "Julho": 7, "Agosto": 8, "Setembro": 9, "Outubro": 10, "Novembro": 11, "Dezembro": 12
    }

    # Construir DataFrame com os dados faltantes
    novos_dados = []
    for ano, meses in dados_es.items():
        for mes_nome, casos in meses.items():
            mes_num = meses_map[mes_nome]
            periodo = pd.Period(f"{ano}-{mes_num:02d}", freq="M")
            novos_dados.append({
               "COD_UF": "ES",
                "periodo": periodo,
                "Quantidade de Casos": casos
            })

    df_novos = pd.DataFrame(novos_dados)

    # Concatenar ao DataFrame original
    df_resultado_final = pd.concat([df_resultado_final, df_novos], ignore_index=True)

# Verificar se os dados foram incluídos
print(df_resultado_final[df_resultado_final["COD_UF"] == "ES"].sort_values("periodo").tail(24))

      Quantidade de Casos COD_UF  periodo
3749                   40     ES  2023-05
3750                   23     ES  2023-06
3751                    9     ES  2023-07
3752                    4     ES  2023-08
3753                    6     ES  2023-09
3754                    6     ES  2023-10
3755                    7     ES  2023-11
3756                    9     ES  2023-12
3757                   62     ES  2024-01
3758                  133     ES  2024-02
3759                  142     ES  2024-03
3760                  105     ES  2024-04
3761                   58     ES  2024-05
3762                   21     ES  2024-06
3763                   11     ES  2024-07
3764                    9     ES  2024-08
3765                    7     ES  2024-09
3766                    6     ES  2024-10
3767                    7     ES  2024-11
3768                   14     ES  2024-12
3551                   20     ES  2025-01
3578                   18     ES  2025-02
3605                    8     ES  

In [29]:
# verifica se há valores não nulos no dataset inicial 
df_resultado_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3769 entries, 0 to 3768
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype    
---  ------               --------------  -----    
 0   Quantidade de Casos  3769 non-null   int64    
 1   COD_UF               3769 non-null   object   
 2   periodo              3769 non-null   period[M]
dtypes: int64(1), object(1), period[M](1)
memory usage: 88.5+ KB


In [ ]:
df_clima = pd.read_csv('..\data\\raw\Dados INMET\ClimaUF_INMET_2010_2025.csv', sep=',')

In [31]:
df_clima

,ano,mes,sigla_uf,precipitacao_media_mensal_uf,temp_max_media_mensal_uf,temp_min_media_mensal_uf,umidade_max_media_mensal_uf,umidade_min_media_mensal_uf,pressao_max_media_mensal_uf,pressao_min_media_mensal_uf
0,2010,1,ES,22.550000,27.831847,26.437251,72.312582,66.147125,993.423711,992.948000
1,2010,1,RN,61.666667,28.522363,27.474542,74.347502,67.765305,1004.199436,1003.660183
2,2010,1,PA,226.082353,27.237584,26.166846,82.357096,77.182888,995.434881,994.826470
3,2010,1,PI,116.044444,27.072992,25.917044,76.544920,71.149372,981.714313,981.142954
4,2010,1,BA,63.142105,26.504802,25.174721,73.001099,66.654399,968.658554,968.091825
...,...,...,...,...,...,...,...,...,...,...
4875,2025,2,SC,128.221053,24.727976,23.489055,79.241245,72.418584,945.386100,944.840855
4876,2025,2,PE,31.685714,26.563042,25.291157,71.701466,66.359144,955.889266,955.384802
4877,2025,2,AP,265.466667,25.913372,25.029640,87.094618,83.921863,1007.094095,1006.497439
4878,2025,2,DF,473.000000,23.672148,22.174650,73.517756,66.349031,895.630057,895.123146


In [ ]:
# Filtra os dados para incluir apenas os anos de 2014 a 2025 para ser compatível com o dataset de dengue
df_clima_2014_2025 = df_clima[(df_clima['ano'] >= 2014)].reset_index(drop=True)

In [33]:
df_clima_2014_2025

,ano,mes,sigla_uf,precipitacao_media_mensal_uf,temp_max_media_mensal_uf,temp_min_media_mensal_uf,umidade_max_media_mensal_uf,umidade_min_media_mensal_uf,pressao_max_media_mensal_uf,pressao_min_media_mensal_uf
0,2014,1,GO,121.920000,24.900063,23.434604,74.407797,66.537762,931.700011,931.129053
1,2014,1,MG,74.600000,24.712463,23.182942,72.561233,65.200438,932.838880,932.343591
2,2014,1,SC,198.484211,23.582267,22.358678,82.252187,76.203870,946.493258,945.962812
3,2014,1,PE,20.850000,27.029737,25.672423,62.156436,56.535063,964.978248,964.491368
4,2014,1,PA,203.655556,26.739380,25.624339,80.438442,75.100748,996.865446,996.252812
...,...,...,...,...,...,...,...,...,...,...
3586,2025,2,SC,128.221053,24.727976,23.489055,79.241245,72.418584,945.386100,944.840855
3587,2025,2,PE,31.685714,26.563042,25.291157,71.701466,66.359144,955.889266,955.384802
3588,2025,2,AP,265.466667,25.913372,25.029640,87.094618,83.921863,1007.094095,1006.497439
3589,2025,2,DF,473.000000,23.672148,22.174650,73.517756,66.349031,895.630057,895.123146


In [34]:
# Cria a coluna 'periodo' no formato period e remove as colunas 'ano' e 'mes' para facilitar o merge posterior
# Renomeia a coluna 'sigla_uf' para 'COD_UF' para facilitar o merge posterior
df_clima_2014_2025["periodo"] = pd.to_datetime(dict(year=df_clima_2014_2025["ano"], month=df_clima_2014_2025["mes"], day=1)).dt.to_period("M")
df_clima_2014_2025.drop(columns=['ano', 'mes'], inplace=True)
df_clima_2014_2025.rename(columns={'sigla_uf': 'COD_UF'}, inplace=True)

In [35]:
df_clima_2014_2025.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3591 entries, 0 to 3590
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype    
---  ------                        --------------  -----    
 0   COD_UF                        3591 non-null   object   
 1   precipitacao_media_mensal_uf  3542 non-null   float64  
 2   temp_max_media_mensal_uf      3566 non-null   float64  
 3   temp_min_media_mensal_uf      3566 non-null   float64  
 4   umidade_max_media_mensal_uf   3552 non-null   float64  
 5   umidade_min_media_mensal_uf   3552 non-null   float64  
 6   pressao_max_media_mensal_uf   3566 non-null   float64  
 7   pressao_min_media_mensal_uf   3566 non-null   float64  
 8   periodo                       3591 non-null   period[M]
dtypes: float64(7), object(1), period[M](1)
memory usage: 252.6+ KB


In [36]:
df_merge = pd.merge(df_resultado_final, df_clima_2014_2025, on=['periodo','COD_UF'], how='inner')

In [37]:
df_merge.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3584 entries, 0 to 3583
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype    
---  ------                        --------------  -----    
 0   Quantidade de Casos           3584 non-null   int64    
 1   COD_UF                        3584 non-null   object   
 2   periodo                       3584 non-null   period[M]
 3   precipitacao_media_mensal_uf  3535 non-null   float64  
 4   temp_max_media_mensal_uf      3559 non-null   float64  
 5   temp_min_media_mensal_uf      3559 non-null   float64  
 6   umidade_max_media_mensal_uf   3545 non-null   float64  
 7   umidade_min_media_mensal_uf   3545 non-null   float64  
 8   pressao_max_media_mensal_uf   3559 non-null   float64  
 9   pressao_min_media_mensal_uf   3559 non-null   float64  
dtypes: float64(7), int64(1), object(1), period[M](1)
memory usage: 280.1+ KB


In [38]:
df_merge.head()

,Quantidade de Casos,COD_UF,periodo,precipitacao_media_mensal_uf,temp_max_media_mensal_uf,temp_min_media_mensal_uf,umidade_max_media_mensal_uf,umidade_min_media_mensal_uf,pressao_max_media_mensal_uf,pressao_min_media_mensal_uf
0,479,RO,2014-01,363.350000,25.335457,24.356153,87.366782,81.982618,973.207926,972.182927
1,521,AC,2014-01,306.200000,26.168398,25.172873,86.335013,80.105127,988.055018,987.365278
2,1537,AM,2014-01,243.305882,26.704184,25.695380,84.414789,79.235912,1003.842685,1003.189520
3,169,RR,2014-01,3.600000,28.693414,27.389785,63.338710,58.205645,1001.820296,1001.218011
4,1085,PA,2014-01,203.655556,26.739380,25.624339,80.438442,75.100748,996.865446,996.252812


In [39]:
df_merge.tail()

,Quantidade de Casos,COD_UF,periodo,precipitacao_media_mensal_uf,temp_max_media_mensal_uf,temp_min_media_mensal_uf,umidade_max_media_mensal_uf,umidade_min_media_mensal_uf,pressao_max_media_mensal_uf,pressao_min_media_mensal_uf
3579,11,ES,2024-07,11.820000,23.008296,21.486787,73.171378,67.371361,990.506774,990.045336
3580,9,ES,2024-08,33.350000,22.199937,20.858429,73.693712,68.202189,993.334688,992.833161
3581,7,ES,2024-09,23.266667,24.125178,22.796199,70.531996,65.142236,991.011829,990.485145
3582,6,ES,2024-10,180.733333,24.288714,23.228466,76.642496,71.943731,988.520179,987.965659
3583,7,ES,2024-11,196.066667,24.407176,23.308276,80.226010,74.910066,986.892281,986.394688


In [40]:
df_merge = df_merge.sort_values(by=['periodo', 'COD_UF']).reset_index(drop=True)

In [41]:
df_merge

,Quantidade de Casos,COD_UF,periodo,precipitacao_media_mensal_uf,temp_max_media_mensal_uf,temp_min_media_mensal_uf,umidade_max_media_mensal_uf,umidade_min_media_mensal_uf,pressao_max_media_mensal_uf,pressao_min_media_mensal_uf
0,521,AC,2014-01,306.200000,26.168398,25.172873,86.335013,80.105127,988.055018,987.365278
1,860,AL,2014-01,31.033333,27.146483,25.862634,73.222670,67.688172,1000.114113,999.650627
2,1537,AM,2014-01,243.305882,26.704184,25.695380,84.414789,79.235912,1003.842685,1003.189520
3,34,AP,2014-01,320.200000,25.811156,24.882997,87.120968,84.208333,1010.215188,1009.659610
4,1429,BA,2014-01,48.005405,26.084308,24.723933,73.336720,66.922169,970.333907,969.846090
...,...,...,...,...,...,...,...,...,...,...
3579,1753,RS,2025-02,105.374359,25.859602,24.442216,77.181434,70.757180,970.803050,970.247375
3580,1611,SC,2025-02,128.221053,24.727976,23.489055,79.241245,72.418584,945.386100,944.840855
3581,46,SE,2025-02,66.666667,27.294540,25.997760,77.757776,70.594630,999.330643,998.872102
3582,162274,SP,2025-02,165.083333,26.164288,24.745914,78.575237,71.490467,947.913471,947.377720


In [ ]:
caminho_salvar_merge = '..\data\\raw\dados_dengue(SINAN)_merge_clima(INMET).csv'
df_merge.to_csv(caminho_salvar_merge, index=False)
print(f"\nResultado final salvo em: {caminho_salvar_merge}")


Resultado final salvo em: C:\Users\55219\Desktop\Lucas\TI\Arquivos\Projetos_aprendizado\TechChallenge_Fase3_Dengue\data\raw\dados_dengue(SINAN)_merge_clima(INMET).csv


In [44]:
# Inclusão de Dados de Saneamento do Painel de Saneamento https://www.painelsaneamento.org.br
import openpyxl
import requests
import io

# Mapeamento de UFs e Códigos IBGE (Inverso do que foi feito anteriormente)
mapa_uf_ibge = {
    'AC': 12, 'AL': 27, 'AP': 16, 'AM': 13, 'BA': 29, 'CE': 23, 'DF': 53,
    'ES': 32, 'GO': 52, 'MA': 21, 'MT': 51, 'MS': 50, 'MG': 31, 'PA': 15,
    'PB': 25, 'PR': 41, 'PE': 26, 'PI': 22, 'RN': 24, 'RS': 43, 'RJ': 33,
    'RO': 11, 'RR': 14, 'SC': 42, 'SP': 35, 'SE': 28, 'TO': 17
}

# Atributos Desejados 
indicadores_desejados = [
    'População total (pessoas) (IBGE)',
    'População urbana total (pessoas) (SINISA)',
    'Número de pessoas por moradia (pessoas por moradia) (IBGE)',
    'Número de pessoas por moradia (área urbana) (pessoas por moradia) (IBGE)',
    'Densidade demográfica (pessoas por km²) (Pessoas por km²) (IBGE)',
    'Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SINISA)',
    'Parcela da população total que mora em domicílios sem acesso à água tratada (% da população) (SINISA)',
    'Índice de esgoto tratado referido à água consumida (%) (SNIS)',
    'Despesas per capita das famílias com saneamento, em R$ a preços de 2024 (deflator IPCA, item água e esgoto) (R$ per capita a preços de 2024) (SINISA)',
    'Internações por doenças associadas à falta de saneamento (Número de internações) (DATASUS)',
    'Taxa de incidência de internações por doenças associadas à falta de saneamento (Internações por 10 mil habitantes) (DATASUS)'
]

# Script de Download e Processamento
lista_dfs_processados = []
base_url_download = "https://www.painelsaneamento.org.br/explore/localidade?SE%5Bl%5D={}&SE%5Bo%5D=e"

print("Iniciando o download e consolidação automática dos dados...")

for uf_sigla, cod_uf in mapa_uf_ibge.items():
    url_download = base_url_download.format(cod_uf)
    print(f"\nBaixando dados para a UF: {uf_sigla} (Código: {cod_uf})...")


    # Faz o download do arquivo Excel para cada UF, processa e filtra apenas os indicadores desejados.
    # O arquivo é lido direto da memória, transposto, reorganizado e convertido para formato longo.
    # Apenas os indicadores relevantes são mantidos e adicionados à lista final.
    try:
        response = requests.get(url_download, timeout=30)
        response.raise_for_status()

        arquivo_excel_em_memoria = io.BytesIO(response.content) # Salvamento do arquivo em memória
        df_largo = pd.read_excel(arquivo_excel_em_memoria, header=2)

        df_largo.rename(columns={df_largo.columns[0]: 'Indicador'}, inplace=True)
        df_largo.dropna(subset=['Indicador'], inplace=True)
        df_largo.drop_duplicates(subset=['Indicador'], keep='first', inplace=True)
        df_largo.set_index('Indicador', inplace=True)

        df_transposto = df_largo.transpose()
        df_transposto.index.name = 'Ano'
        df_longo = df_transposto.reset_index()

        df_longo['COD_UF'] = uf_sigla

        df_melted = df_longo.melt(
            id_vars=['COD_UF', 'Ano'],
            var_name='Indicador',
            value_name='Valor'
        ) 

        df_filtrado = df_melted[df_melted['Indicador'].isin(indicadores_desejados)]
        lista_dfs_processados.append(df_filtrado)
        print(f"  - Dados de {uf_sigla} processados e filtrados com sucesso.")

    except Exception as e:
        print(f"  ERRO ao processar a UF {uf_sigla}: {e}")

# Consolidação e Limpeza Final
if lista_dfs_processados:
    df_saneamento_completo = pd.concat(lista_dfs_processados, ignore_index=True)

    # Limpeza primária dos valores
    df_saneamento_completo['Valor'] = df_saneamento_completo['Valor'].astype(str)
    df_saneamento_completo['Valor'] = df_saneamento_completo['Valor'].str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
    df_saneamento_completo['Valor'] = pd.to_numeric(df_saneamento_completo['Valor'], errors='coerce')

    # Pivotagem da tabela para agrupar por COD_UF e Ano e ter como atributpos os indicadores
    df_final = df_saneamento_completo.pivot_table(
        index=['COD_UF', 'Ano'],
        columns='Indicador',
        values='Valor'
    ).reset_index()
    df_final.columns.name = None

    print("\n--- Processamento Concluído! A aplicar correção de escala... ---")

    # Correção de escala devido a inconsistências nos dados originais
    colunas_para_dividir_100 = [
        'Número de pessoas por moradia (pessoas por moradia) (IBGE)',
        'Número de pessoas por moradia (área urbana) (pessoas por moradia) (IBGE)',
        'Densidade demográfica (pessoas por km²) (Pessoas por km²) (IBGE)',
        'Índice de esgoto tratado referido à água consumida (%) (SNIS)',
        'Despesas per capita das famílias com saneamento, em R$ a preços de 2024 (deflator IPCA, item água e esgoto) (R$ per capita a preços de 2024) (SINISA)'
    ]

    colunas_para_dividir_10 = [
        'Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SINISA)',
        'Parcela da população total que mora em domicílios sem acesso à água tratada (% da população) (SINISA)'
    ]

    coluna_para_dividir_1000 = 'Taxa de incidência de internações por doenças associadas à falta de saneamento (Internações por 10 mil habitantes) (DATASUS)'

    for coluna in colunas_para_dividir_100:
        if coluna in df_final.columns:
            df_final[coluna] = df_final[coluna] / 100

    for coluna in colunas_para_dividir_10:
        if coluna in df_final.columns:
            df_final[coluna] = df_final[coluna] / 10

    if coluna_para_dividir_1000 in df_final.columns:
        df_final[coluna_para_dividir_1000] = df_final[coluna_para_dividir_1000] / 1000

    print("Correção de escala aplicada com sucesso!")
    # ------------------------------------

    print("\n--- DataFrame Final Consolidado e Corrigido ---")
    print(df_final.head())

else:
    print("\nProcessamento concluído, mas nenhum dado foi gerado.")

Iniciando o download e consolidação automática dos dados...

Baixando dados para a UF: AC (Código: 12)...
  - Dados de AC processados e filtrados com sucesso.

Baixando dados para a UF: AL (Código: 27)...
  - Dados de AL processados e filtrados com sucesso.

Baixando dados para a UF: AP (Código: 16)...
  - Dados de AP processados e filtrados com sucesso.

Baixando dados para a UF: AM (Código: 13)...
  - Dados de AM processados e filtrados com sucesso.

Baixando dados para a UF: BA (Código: 29)...
  - Dados de BA processados e filtrados com sucesso.

Baixando dados para a UF: CE (Código: 23)...
  - Dados de CE processados e filtrados com sucesso.

Baixando dados para a UF: DF (Código: 53)...
  - Dados de DF processados e filtrados com sucesso.

Baixando dados para a UF: ES (Código: 32)...
  - Dados de ES processados e filtrados com sucesso.

Baixando dados para a UF: GO (Código: 52)...
  - Dados de GO processados e filtrados com sucesso.

Baixando dados para a UF: MA (Código: 21)...
  -

In [45]:
df_final[df_final['COD_UF'] == 'SP']

,COD_UF,Ano,Densidade demográfica (pessoas por km²) (Pessoas por km²) (IBGE),"Despesas per capita das famílias com saneamento, em R$ a preços de 2024 (deflator IPCA, item água e esgoto) (R$ per capita a preços de 2024) (SINISA)",Internações por doenças associadas à falta de saneamento (Número de internações) (DATASUS),Número de pessoas por moradia (pessoas por moradia) (IBGE),Número de pessoas por moradia (área urbana) (pessoas por moradia) (IBGE),Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SINISA),Parcela da população total que mora em domicílios sem acesso à água tratada (% da população) (SINISA),População total (pessoas) (IBGE),População urbana total (pessoas) (SINISA),Taxa de incidência de internações por doenças associadas à falta de saneamento (Internações por 10 mil habitantes) (DATASUS)
350,SP,2010,166.19,698.45,35645.0,3.21,3.07,13.9,4.3,41252160.0,37943252.0,8.641
351,SP,2011,167.54,736.97,25883.0,3.15,0.31,13.3,4.3,41587182.0,39386041.0,6.224
352,SP,2012,168.81,732.39,24425.0,0.31,3.06,12.6,0.4,41901219.0,39816112.0,5.829
353,SP,2013,175.91,703.22,23768.0,3.14,0.31,12.6,4.1,43663669.0,41425753.0,5.443
354,SP,2014,17.74,666.19,25907.0,3.09,3.06,1.2,4.2,44035304.0,41935126.0,5.883
355,SP,2015,178.86,623.87,33312.0,3.03,0.03,11.6,4.4,44396484.0,42247247.0,7.503
356,SP,2016,180.28,637.41,23128.0,2.98,2.96,11.2,4.2,44749699.0,42756931.0,5.168
357,SP,2017,181.67,628.86,16944.0,2.92,0.29,10.3,3.8,45094866.0,42982604.0,3.757
358,SP,2018,183.46,655.97,16873.0,2.88,2.85,10.2,3.8,45538936.0,43460950.0,3.705
359,SP,2019,184.99,638.22,26059.0,2.85,2.84,9.7,3.8,45919049.0,44064368.0,5.675


In [46]:
df_final_2014_2023 = df_final[df_final['Ano'] >=2014]

In [47]:
df_final_2014_2023

,COD_UF,Ano,Densidade demográfica (pessoas por km²) (Pessoas por km²) (IBGE),"Despesas per capita das famílias com saneamento, em R$ a preços de 2024 (deflator IPCA, item água e esgoto) (R$ per capita a preços de 2024) (SINISA)",Internações por doenças associadas à falta de saneamento (Número de internações) (DATASUS),Número de pessoas por moradia (pessoas por moradia) (IBGE),Número de pessoas por moradia (área urbana) (pessoas por moradia) (IBGE),Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SINISA),Parcela da população total que mora em domicílios sem acesso à água tratada (% da população) (SINISA),População total (pessoas) (IBGE),População urbana total (pessoas) (SINISA),Taxa de incidência de internações por doenças associadas à falta de saneamento (Internações por 10 mil habitantes) (DATASUS)
4,AC,2014,4.81,141.07,3459.0,3.65,3.49,8.8,55.4,790101.0,572554.0,43.779
5,AC,2015,0.49,136.53,2167.0,0.36,3.44,87.5,5.3,803513.0,582101.0,26.969
6,AC,2016,4.98,12.75,1938.0,3.54,3.39,87.8,5.2,816687.0,591481.0,2.373
7,AC,2017,5.05,112.91,1320.0,3.49,3.34,89.3,50.9,829619.0,600692.0,15.911
8,AC,2018,0.53,122.85,1367.0,3.54,3.39,89.9,52.9,869265.0,629627.0,15.726
...,...,...,...,...,...,...,...,...,...,...,...,...
373,TO,2019,5.66,445.17,2125.0,2.99,2.88,70.5,16.5,1572866.0,1252038.0,1.351
374,TO,2020,5.73,467.07,1100.0,3.12,3.43,73.1,2.1,1590248.0,1427874.0,6.917
375,TO,2021,5.79,486.64,1231.0,3.08,4.38,70.9,20.4,1607363.0,1455927.0,7.659
376,TO,2022,5.44,567.44,1802.0,2.94,NaN,0.1,5.9,1511460.0,NaN,11.922


In [ ]:
caminho_salvar_saneamento = '..\data\\raw\Dados_Saneamento\dados_saneamento_2014_2023.csv'
df_final_2014_2023.to_csv(caminho_salvar_saneamento, index=False)
print(f"\nResultado final salvo em: {caminho_salvar_saneamento}")


Resultado final salvo em: C:\Users\55219\Desktop\Lucas\TI\Arquivos\Projetos_aprendizado\TechChallenge_Fase3_Dengue\data\raw\Dados_Saneamento\dados_saneamento_2014_2023.csv


In [50]:
df_clima_dengue = pd.read_csv('C:\\Users\\55219\Desktop\Lucas\TI\Arquivos\Projetos_aprendizado\TechChallenge_Fase3_Dengue\data\\raw\dados_dengue(SINAN)_merge_clima(INMET).csv')
df_saneamento = pd.read_csv('C:\\Users\\55219\Desktop\Lucas\TI\Arquivos\Projetos_aprendizado\TechChallenge_Fase3_Dengue\data\\raw\Dados_Saneamento\dados_saneamento_2014_2023.csv')

In [51]:
df_clima_dengue.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3584 entries, 0 to 3583
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Quantidade de Casos           3584 non-null   int64  
 1   COD_UF                        3584 non-null   object 
 2   periodo                       3584 non-null   object 
 3   precipitacao_media_mensal_uf  3535 non-null   float64
 4   temp_max_media_mensal_uf      3559 non-null   float64
 5   temp_min_media_mensal_uf      3559 non-null   float64
 6   umidade_max_media_mensal_uf   3545 non-null   float64
 7   umidade_min_media_mensal_uf   3545 non-null   float64
 8   pressao_max_media_mensal_uf   3559 non-null   float64
 9   pressao_min_media_mensal_uf   3559 non-null   float64
dtypes: float64(7), int64(1), object(2)
memory usage: 280.1+ KB


In [52]:
df_saneamento.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 12 columns):
 #   Column                                                                                                                                                 Non-Null Count  Dtype  
---  ------                                                                                                                                                 --------------  -----  
 0   COD_UF                                                                                                                                                 270 non-null    object 
 1   Ano                                                                                                                                                    270 non-null    int64  
 2   Densidade demográfica (pessoas por km²) (Pessoas por km²) (IBGE)                                                                                       270 non-null    flo

In [53]:
df_clima_dengue['Ano'] = df_clima_dengue['periodo'].astype(str).str.split('-').str[0].astype(int)
df_clima_dengue_saneamento = pd.merge(df_clima_dengue, df_saneamento, on=['Ano', 'COD_UF'], how='left')
df_clima_dengue_saneamento.drop(columns=['Ano'], inplace=True)

In [54]:
display(df_clima_dengue_saneamento[df_clima_dengue_saneamento['COD_UF'] == 'SP'])

,Quantidade de Casos,COD_UF,periodo,precipitacao_media_mensal_uf,temp_max_media_mensal_uf,temp_min_media_mensal_uf,umidade_max_media_mensal_uf,umidade_min_media_mensal_uf,pressao_max_media_mensal_uf,pressao_min_media_mensal_uf,Densidade demográfica (pessoas por km²) (Pessoas por km²) (IBGE),"Despesas per capita das famílias com saneamento, em R$ a preços de 2024 (deflator IPCA, item água e esgoto) (R$ per capita a preços de 2024) (SINISA)",Internações por doenças associadas à falta de saneamento (Número de internações) (DATASUS),Número de pessoas por moradia (pessoas por moradia) (IBGE),Número de pessoas por moradia (área urbana) (pessoas por moradia) (IBGE),Parcela da população total que mora em domicílios sem acesso ao serviço de coleta de esgoto (% da população) (SINISA),Parcela da população total que mora em domicílios sem acesso à água tratada (% da população) (SINISA),População total (pessoas) (IBGE),População urbana total (pessoas) (SINISA),Taxa de incidência de internações por doenças associadas à falta de saneamento (Internações por 10 mil habitantes) (DATASUS)
25,8622,SP,2014-01,114.281481,25.654945,24.108317,72.909421,64.908200,941.355779,940.793196,17.74,666.19,25907.0,3.09,3.06,1.2,4.2,44035304.0,41935126.0,5.883
52,19518,SP,2014-02,97.715385,25.992219,24.458004,67.474113,59.905319,940.749388,940.244697,17.74,666.19,25907.0,3.09,3.06,1.2,4.2,44035304.0,41935126.0,5.883
79,46080,SP,2014-03,116.829630,24.397119,23.039601,77.044962,69.998108,943.943171,943.420052,17.74,666.19,25907.0,3.09,3.06,1.2,4.2,44035304.0,41935126.0,5.883
106,125274,SP,2014-04,78.503704,22.579267,21.326751,77.793980,71.522775,945.189900,944.702260,17.74,666.19,25907.0,3.09,3.06,1.2,4.2,44035304.0,41935126.0,5.883
133,97607,SP,2014-05,43.786207,19.763936,18.416700,75.638548,69.383651,945.962257,945.494626,17.74,666.19,25907.0,3.09,3.06,1.2,4.2,44035304.0,41935126.0,5.883
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3474,15277,SP,2024-09,22.811111,25.315601,23.585261,54.333027,48.580915,950.009914,949.465453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3501,14301,SP,2024-10,120.464516,24.360028,22.985431,70.642985,64.874926,948.309162,947.713949,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3528,16752,SP,2024-11,182.935484,24.236102,22.978686,76.728028,70.958130,944.541191,944.002615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3555,114331,SP,2025-01,159.792593,25.316222,23.922948,78.417827,71.579303,945.897319,945.347440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [55]:
df_clima_dengue_saneamento.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3584 entries, 0 to 3583
Data columns (total 20 columns):
 #   Column                                                                                                                                                 Non-Null Count  Dtype  
---  ------                                                                                                                                                 --------------  -----  
 0   Quantidade de Casos                                                                                                                                    3584 non-null   int64  
 1   COD_UF                                                                                                                                                 3584 non-null   object 
 2   periodo                                                                                                                                                3584 non-null   o

In [ ]:
caminho_salvar_final = '..\data\\raw\dados_dengue_clima_saneamento_2014_2025.csv'
df_clima_dengue_saneamento.to_csv(caminho_salvar_final, index=False)
print(f"\nResultado final salvo em: {caminho_salvar_final}")


Resultado final salvo em: C:\Users\55219\Desktop\Lucas\TI\Arquivos\Projetos_aprendizado\TechChallenge_Fase3_Dengue\data\raw\dados_dengue_clima_saneamento_2014_2025.csv
